In [1]:
%load_ext autoreload
%autoreload 2

from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT
from SIDER_dataset.libraries.feature_evaluation_methods import feature_variance_reduction_scores
from SIDER_dataset.libraries.utils import get_clus_path


In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")

9 datasets


In [3]:
k = 10
random_state = 42
performances = []
ranking_criteria = "MDI"  # or "VAR"
include_original_features_options = [True, False]
training_algorithm = "Jaccard_min"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy"]
max_size = 5
cv_results = []

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criteria}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            training_algorithm,
            "_".join(eval_criteria),
            str(max_size),
        ]
    )
    logging_path = f"XofN_jaccard_min/logs/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        if ranking_criteria == "VAR":
            feature_rankings = feature_variance_reduction_scores(train_dataset[features],
                                                                 train_dataset[path["label_set"]])
        elif ranking_criteria == "MDI":
            feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])
        else:
            raise NotImplementedError

        XofN_groupings, avg_features, gen_XofN_time = generate_XofN_list_multi_jaccard(
            train_dataset,
            feature_rankings,
            max_size,
            logger,
            minimise=True
        )
        if len(XofN_groupings) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = group_features(
                    train_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_test_dataset = group_features(
                    test_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_train_dataset.to_csv(f"XofN_jaccard_min/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"XofN_jaccard_min/tmp/test_dataset.csv", index=False)

                training_start = time.perf_counter()
                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "XofN_jaccard_min/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"XofN_jaccard_min/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            ['averageAUROC', 'HammingLoss', 'SubsetAccuracy', 'nodes', 'leaves', 'groups',
             'avg_group_features', 'gen_XofN_time', 'training_time']].mean().reset_index()
        print(averages)
        cv_results.append(averages)
        performances = []
# paths[0] - features[:10] desktop 0.29m
# laptop ??m
# desktop 12h


--- Running with label:'['se_C0027497', 'se_C0018681', 'se_C0011603', 'se_C0015230', 'se_C0042963', 'se_C0012833', 'se_C0027769', 'se_C0020580', 'se_C0014457', 'se_C0017181', 'se_C0151763', 'se_C0038358']' training_algorithm:'Jaccard_min' eval_criterion:'['averageAUROC', 'HammingLoss', 'SubsetAccuracy']' max_size:'5' ---
Logs can be found in XofN_jaccard_min/logs/CPI+fingerprint_all_Jaccard_min_averageAUROC_HammingLoss_SubsetAccuracy_5_logs.txt.

Fold 1/10 (CPI+fingerprint_all 1/9)

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 2147/2147 [10:13<00:00,  3.50feat/s] 


XofN_groups: [['cpi_9606.ENSP00000337915', 'f_413', 'f_262', 'f_151', 'f_635'], ['f_374', 'cpi_9606.ENSP00000268695', 'cpi_9606.ENSP00000414334', 'f_768', 'cpi_9606.ENSP00000359424'], ['f_308', 'cpi_9606.ENSP00000355192', 'f_616', 'cpi_9606.ENSP00000351957', 'cpi_9606.ENSP00000374323'], ['f_20', 'cpi_9606.ENSP00000333212', 'cpi_9606.ENSP00000291700', 'cpi_9606.ENSP00000295454', 'f_194'], ['f_3', 'cpi_9606.ENSP00000023897', 'cpi_9606.ENSP00000428864', 'cpi_9606.ENSP00000337335', 'cpi_9606.ENSP00000286479'], ['f_299', 'cpi_9606.ENSP00000342850', 'cpi_9606.ENSP00000323867', 'cpi_9606.ENSP00000367851', 'cpi_9606.ENSP00000360493'], ['f_452', 'f_25', 'cpi_9606.ENSP00000221421', 'cpi_9606.ENSP00000259396', 'f_831'], ['f_672', 'cpi_9606.ENSP00000409007', 'cpi_9606.ENSP00000321584', 'f_292', 'f_845'], ['f_23', 'cpi_9606.ENSP00000241052', 'f_118', 'cpi_9606.ENSP00000349320', 'cpi_9606.ENSP00000356094'], ['f_393', 'cpi_9606.ENSP00000292427', 'cpi_9606.ENSP00000363812', 'cpi_9606.ENSP00000380318',

🔄 Processing features: 100%|██████████| 2147/2147 [10:14<00:00,  3.49feat/s] 


XofN_groups: [['cpi_9606.ENSP00000337915', 'f_413', 'f_262', 'f_428', 'cpi_9606.ENSP00000357669'], ['f_308', 'cpi_9606.ENSP00000288139', 'f_616', 'cpi_9606.ENSP00000359334', 'cpi_9606.ENSP00000352035'], ['f_374', 'cpi_9606.ENSP00000312304', 'cpi_9606.ENSP00000376178', 'cpi_9606.ENSP00000297268', 'cpi_9606.ENSP00000295452'], ['f_20', 'cpi_9606.ENSP00000229264', 'cpi_9606.ENSP00000262352', 'cpi_9606.ENSP00000291700', 'cpi_9606.ENSP00000295454'], ['f_406', 'cpi_9606.ENSP00000355192', 'cpi_9606.ENSP00000376914', 'cpi_9606.ENSP00000335592', 'cpi_9606.ENSP00000369375'], ['f_393', 'cpi_9606.ENSP00000342850', 'cpi_9606.ENSP00000323867', 'f_151', 'cpi_9606.ENSP00000367851'], ['f_346', 'cpi_9606.ENSP00000279178', 'cpi_9606.ENSP00000357068', 'cpi_9606.ENSP00000395653', 'f_629'], ['f_299', 'cpi_9606.ENSP00000292427', 'cpi_9606.ENSP00000363812', 'cpi_9606.ENSP00000357336', 'cpi_9606.ENSP00000331912'], ['f_451', 'cpi_9606.ENSP00000312286', 'cpi_9606.ENSP00000254976', 'cpi_9606.ENSP00000320025', 'cpi

...

XofN_groups: [['f_374', 'f_831', 'f_27', 'f_771', 'f_675'], ['f_20', 'f_792', 'f_629', 'f_328', 'f_194'], ['f_308', 'f_616', 'f_785', 'f_726', 'f_327'], ['f_299', 'f_866', 'f_31', 'f_214', 'f_622'], ['f_366', 'f_789', 'f_428', 'f_635', 'f_297'], ['f_23', 'f_413', 'f_631', 'f_118', 'f_47'], ['f_516', 'f_350', 'f_587', 'f_202', 'f_768'], ['f_24', 'f_46', 'f_408', 'f_215', 'f_151'], ['f_406', 'f_791', 'f_836', 'f_26', 'f_788'], ['f_3', 'f_409', 'f_834', 'f_330', 'f_830'], ['f_571', 'f_736', 'f_529', 'f_554', 'f_429'], ['f_672', 'f_292', 'f_845', 'f_767', 'f_298'], ['f_19', 'f_670', 'f_510', 'f_773', 'f_460'], ['f_393', 'f_861', 'f_526', 'f_229', 'f_497'], ['f_639', 'f_417', 'f_750', 'f_511', 'f_568'], ['f_15', 'f_839', 'f_719', 'f_509', 'f_764'], ['f_452', 'f_25', 'f_722', 'f_757', 'f_43'], ['f_287', 'f_35', 'f_799', 'f_740', 'f_609'], ['f_346', 'f_725', 'f_749', 'f_427', 'f_455'], ['f_380', 'f_729', 'f_820', 'f_401', 'f_436'], ['f_440', 'f_728', 'f_813', 'f_388', 'f_715'], ['f_37', 'f_78

🔄 Processing features: 100%|██████████| 540/540 [01:28<00:00,  6.13feat/s]


XofN_groups: [['f_374', 'f_834', 'f_768', 'f_31', 'f_202'], ['f_20', 'f_792', 'f_629', 'f_194', 'f_635'], ['f_308', 'f_616', 'f_831', 'f_785', 'f_328'], ['f_346', 'f_726', 'f_428', 'f_799', 'f_297'], ['f_366', 'f_789', 'f_327', 'f_510', 'f_460'], ['f_516', 'f_47', 'f_413', 'f_27', 'f_771'], ['f_23', 'f_687', 'f_43', 'f_736', 'f_529'], ['f_672', 'f_292', 'f_845', 'f_830', 'f_815'], ['f_639', 'f_417', 'f_767', 'f_131', 'f_25'], ['f_440', 'f_791', 'f_788', 'f_722', 'f_118'], ['f_24', 'f_622', 'f_408', 'f_216', 'f_151'], ['f_452', 'f_757', 'f_729', 'f_866', 'f_719'], ['f_3', 'f_676', 'f_836', 'f_511', 'f_215'], ['f_697', 'f_35', 'f_725', 'f_749', 'f_436'], ['f_15', 'f_861', 'f_773', 'f_631', 'f_554'], ['f_571', 'f_526', 'f_820', 'f_454', 'f_214'], ['f_406', 'f_728', 'f_778', 'f_26', 'f_409'], ['f_380', 'f_330', 'f_587', 'f_650', 'f_764'], ['f_299', 'f_840', 'f_229', 'f_670', 'f_583'], ['f_656', 'f_706', 'f_675', 'f_782', 'f_427'], ['f_287', 'f_207', 'f_509', 'f_448', 'f_609'], ['f_393', 'f

🔄 Processing features: 100%|██████████| 540/540 [01:27<00:00,  6.15feat/s]


XofN_groups: [['f_374', 'f_768', 'f_27', 'f_771', 'f_629'], ['f_20', 'f_792', 'f_194', 'f_497', 'f_214'], ['f_23', 'f_408', 'f_207', 'f_31', 'f_635'], ['f_366', 'f_726', 'f_428', 'f_820', 'f_510'], ['f_346', 'f_789', 'f_749', 'f_460', 'f_799'], ['f_24', 'f_815', 'f_413', 'f_43', 'f_407'], ['f_516', 'f_47', 'f_292', 'f_587', 'f_834'], ['f_308', 'f_616', 'f_831', 'f_722', 'f_25'], ['f_287', 'f_195', 'f_166', 'f_118', 'f_529'], ['f_672', 'f_845', 'f_729', 'f_297', 'f_167'], ['f_15', 'f_866', 'f_836', 'f_719', 'f_436'], ['f_639', 'f_427', 'f_830', 'f_131', 'f_330'], ['f_393', 'f_151', 'f_622', 'f_785', 'f_675'], ['f_452', 'f_757', 'f_773', 'f_518', 'f_466'], ['f_440', 'f_791', 'f_788', 'f_526', 'f_388'], ['f_571', 'f_752', 'f_736', 'f_327', 'f_511'], ['f_614', 'f_767', 'f_26', 'f_350', 'f_328'], ['f_380', 'f_554', 'f_650', 'f_813', 'f_706'], ['f_406', 'f_728', 'f_725', 'f_583', 'f_202'], ['f_299', 'f_861', 'f_764', 'f_229', 'f_670'], ['f_3', 'f_477', 'f_401', 'f_509', 'f_471'], ['f_37', 'f

In [4]:
# All results
save_path = "XofN_jaccard_min/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,True,with_org,fingerprint_frequent,0.541951,0.226448,0.512931,47.4,24.2,108.0,5.000000,87.911732,1.488730
1,True,no_org,fingerprint_frequent,0.550990,0.233914,0.500773,39.6,20.3,108.0,5.000000,87.911732,0.727399
2,False,with_org,fingerprint_frequent,0.573193,0.278743,0.359495,599.0,300.0,108.0,5.000000,87.911732,1.488730
3,False,no_org,fingerprint_frequent,0.575128,0.286729,0.362462,608.2,304.6,108.0,5.000000,87.911732,0.727399
4,True,no_org,fingerprint_diverse,0.500000,0.181567,0.425757,1.0,1.0,108.0,5.000000,87.787998,0.743740
5,False,with_org,fingerprint_diverse,0.567529,0.268308,0.229548,777.8,389.4,108.0,5.000000,87.787998,1.584081
6,False,no_org,fingerprint_diverse,0.562458,0.276895,0.227272,805.4,403.2,108.0,5.000000,87.787998,0.743740
7,True,with_org,fingerprint_diverse,0.516977,0.178788,0.437122,3.6,2.3,108.0,5.000000,87.787998,1.584081
8,True,with_org,fingerprint_all,0.514791,0.198918,0.110685,3.2,2.1,108.0,5.000000,87.712154,1.635671
9,True,no_org,fingerprint_all,0.500000,0.200827,0.104580,1.2,1.1,108.0,5.000000,87.712154,0.745348


In [5]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_jaccard_min/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
1,no_org,fingerprint_frequent,0.551,0.234,0.501,39.6; 20.3,108.0; 5.0,0.0,87.9; 0.7
0,with_org,fingerprint_frequent,0.542,0.226,0.513,47.4; 24.2,108.0; 5.0,0.0,87.9; 1.5
4,no_org,fingerprint_diverse,0.500,0.182,0.426,1.0; 1.0,108.0; 5.0,0.0,87.8; 0.7
7,with_org,fingerprint_diverse,0.517,0.179,0.437,3.6; 2.3,108.0; 5.0,0.0,87.8; 1.6
9,no_org,fingerprint_all,0.500,0.201,0.105,1.2; 1.1,108.0; 5.0,0.0,87.7; 0.7
8,with_org,fingerprint_all,0.515,0.199,0.111,3.2; 2.1,108.0; 5.0,0.0,87.7; 1.6
13,no_org,CPI_frequent,0.508,0.214,0.543,12.0; 6.5,322.0; 5.0,0.0,411.8; 0.9
12,with_org,CPI_frequent,0.538,0.215,0.539,19.2; 10.1,322.0; 5.0,0.0,411.8; 2.9
17,no_org,CPI_diverse,0.531,0.192,0.407,6.6; 3.8,322.0; 5.0,0.0,411.1; 1.0
16,with_org,CPI_diverse,0.559,0.195,0.401,14.0; 7.5,322.0; 5.0,0.0,411.1; 3.0
